In [ ]:
import numpy as np
import soundfile as sf
import pyworld as pw
import librosa


import matplotlib.pyplot as plt
import librosa.display
from analysis.visualize import create_spectrogram_plot
from IPython.display import Audio, display
from pathlib import Path

PROJECT_ROOT = Path.cwd()/".."/".."

print(PROJECT_ROOT)
source_audio_path = PROJECT_ROOT / "data/VocalSet/FULL/female1/long_tones/forte/f1_long_forte_a.wav"
violin_audio_path = PROJECT_ROOT / "data/150_Gm_BllywdVln_SP_39_01.wav"
output_audio_path = PROJECT_ROOT / "data/voice_to_violin.wav"

display(Audio(filename=source_audio_path))
display(Audio(filename=violin_audio_path))

In [ ]:


#########################
# 1) Load source (voice) and target (violin) audio
x_src, fs = sf.read(source_audio_path)     # mono, float64
x_vl,  fs_vl = sf.read(violin_audio_path)    # violin recording for timbre model
if x_vl.ndim == 2:
    # convert to mono by averaging channels (or choose one channel)
    x_vl = np.mean(x_vl, axis=1)
assert fs == fs_vl, "Sampling rates must match"

#########################
# 2) Analyse the violin to get its “timbre model” (envelope + aperiodicity)
f0_vl, t_vl = pw.dio(x_vl, fs)             # coarse F0 (not strictly needed)
f0_vl  = pw.stonemask(x_vl, f0_vl, t_vl, fs)  # refined F0
sp_vl  = pw.cheaptrick(x_vl, f0_vl, t_vl, fs) # spectral envelope of violin
ap_vl  = pw.d4c(x_vl, f0_vl, t_vl, fs)        # aperiodicity (noise) envelope

# Build a “global violin timbre model” by averaging over time:
sp_mean_vl = np.mean(sp_vl, axis=0, keepdims=True)
ap_mean_vl = np.mean(ap_vl, axis=0, keepdims=True)

#########################
# 3) Analyse the voice (source) you want to “violin-ize”
f0_src, t_src = pw.dio(x_src, fs)
f0_src  = pw.stonemask(x_src, f0_src, t_src, fs)
sp_src  = pw.cheaptrick(x_src, f0_src, t_src, fs)
ap_src  = pw.d4c(x_src, f0_src, t_src, fs)



In [ ]:
# Resynthesize the source audio to check the analysis-synthesis process
y_src_resyn = pw.synthesize(f0_src, sp_src, ap_src, fs)

# Plot original vs. resynthesized source spectrogram

create_spectrogram_plot([x_src,y_src_resyn], fs)
# Also, let's hear it
print("Original source audio:")
display(Audio(data=x_src, rate=fs))
print("Resynthesized source audio:")
display(Audio(data=y_src_resyn, rate=fs))

In [ ]:
y = pw.synthesize(
    f0_src,
    np.repeat(sp_mean_vl, sp_src.shape[0], axis=0),# ,  # per-frame violin envelope
    np.repeat(ap_mean_vl, ap_src.shape[0], axis=0),  # per-frame aperiodicity envelope (zeros for now)
    fs
)

#########################
# 5) Save
sf.write(output_audio_path, y, fs)

In [ ]:
display(Audio(filename=output_audio_path, rate=fs))
display(Audio(filename=source_audio_path, rate=fs))
display(Audio(filename=violin_audio_path, rate=fs))

In [ ]:


# Create a figure for the spectrograms
fig, ax = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(10, 12))

# 1. Spectrogram for the source audio (voice)
D_src = librosa.amplitude_to_db(np.abs(librosa.stft(x_src)), ref=np.max)
librosa.display.specshow(D_src, y_axis='log', x_axis='time', sr=fs, ax=ax[0])
ax[0].set_title('Source (Voice) Spectrogram')
ax[0].set_ylabel('Frequency [Hz]')

# 2. Spectrogram for the target audio (violin)
D_vl = librosa.amplitude_to_db(np.abs(librosa.stft(x_vl)), ref=np.max)
librosa.display.specshow(D_vl, y_axis='log', x_axis='time', sr=fs, ax=ax[1])
ax[1].set_title('Target (Violin) Spectrogram')
ax[1].set_ylabel('Frequency [Hz]')

# 3. Spectrogram for the synthesized audio
D_y = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
img = librosa.display.specshow(D_y, y_axis='log', x_axis='time', sr=fs, ax=ax[2])
ax[2].set_title('Synthesized (Voice-to-Violin) Spectrogram')
ax[2].set_xlabel('Time [s]')
ax[2].set_ylabel('Frequency [Hz]')

# Add a colorbar to the figure
# fig.colorbar(img, ax=ax, format='%+2.0f dB')

# Adjust layout and display the plot
plt.tight_layout()
plt.show()